# 15 · CP5 repair: token-level deletion before fusion (L4, one session)

**L4 runtime**, one session. About **1.5 compute units** at 2.2 CU/h (estimate). Type Colab's balance into `CU_BALANCE_START` in cell 2 before Run all.

**Why:** in notebook 14 (19 Nov) the pixel deletion test was confounded. Blurring a region is itself a local edit, so P(edited) stayed about 0.9 in every removal order, and deletion and insertion came out almost equal. CP5 was recorded as **FAIL**. The mask decoder is now the primary evidence map and Grad-CAM++ secondary (Dhanush, 24 Nov 2026).

**The pre-registered test** (only the test changes, not the model; rules fixed in the spec card before this run):
- **What is removed:** on the best tile of each of the 300 `P6_deletion` images, fused-grid positions are removed in attribution order. At each removed position, **both** the semantic patch token and the trace token are replaced by their per-tile mean token.
- **What is re-run:** fusion + gate + class head only. CLIP and the trace CNN run once, and no pixel changes.
- **The class token** of CLIP is **not** removed, so the curves need not reach 0.
- **Orders:** the **oracle** (true edited share per token), the **decoder** map, **Grad-CAM++**, and the mean of **10 seeded random** orders.
- **Validity first:** the oracle AUC must be below random (one-sided paired Wilcoxon, p < 0.01). Otherwise the test is reported as **uninformative** for both maps.
- **Per map:** the map's AUC is below random with p < 0.01 → PASS, else FAIL.
- **Randomization** is rerun on the 30 validation and 50 P6 tiles of notebook 14. **SSIM < 0.5 stays the criterion**; Spearman rank correlation is reported beside it.

**P6 cache:**
- If `Paper 1/runs/cache/p6_images.tar` exists, it is copied and checked against its SHA-256 manifest.
- Otherwise the 1,500 P6 images are staged as in notebook 14 and saved into that tar, so later notebooks need no 7z.

**Results** (send to Claude):
- `Paper 1/runs/cp5/cp5_token_deletion_seed42.json`;
- `p6_token_deletion_seed42.csv`;
- `cp5_token_curves.png`;
- `compute_cp5_token.json`.

In [ ]:
# Cell 1: mount Drive, get the code, install it
import os, subprocess, sys

DRY = os.environ.get('EVIGROUND_DRY_RUN') == '1'   # CI / local test: fake sources, temp folders
REPO_URL = 'github.com/DHANUSHKUMMARI/Paper1.git'
BRANCH = 'main'                                    # 'claude/nice-curie-3tzxzr' before the merge

if DRY:
    REPO = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
else:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    REPO = '/content/Paper1'
    if not os.path.exists(REPO):
        try:
            tok = userdata.get('GITHUB_TOKEN')       # never printed
        except Exception:
            tok = None
        url = f'https://{tok}@{REPO_URL}' if tok else f'https://{REPO_URL}'
        r = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, REPO],
                           capture_output=True, text=True)
        if r.returncode != 0:                        # stderr is not shown: it can contain the URL
            raise RuntimeError('git clone failed: check GITHUB_TOKEN (Secrets panel) and BRANCH')
        subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'],
                       check=True)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO + '[train]'], check=True)
    # Colab ships torchao 0.10; peft refuses a torchao older than 0.16 with an ImportError when it
    # adds the LoRA layers, although EviGround never uses torchao: remove an old one.
    import importlib, importlib.metadata as _md
    from packaging.version import Version
    try:
        if Version(_md.version('torchao')) < Version('0.16.0'):
            subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=True)
            importlib.invalidate_caches()
    except _md.PackageNotFoundError:
        pass

sys.path.insert(0, os.path.join(REPO, 'src'))
# Re-running in a runtime that already imported eviground would keep the OLD code in memory
# although git pulled the new one: drop it so the next import reads the files on disk.
for _name in [m for m in sys.modules if m == 'eviground' or m.startswith('eviground.')]:
    del sys.modules[_name]
print('repo:', REPO, '| branch:', BRANCH, '| dry run:', DRY)

In [ ]:
# Cell 2: settings (type CU_BALANCE_START first)
import csv, datetime, json, logging, shutil, subprocess, tempfile, time, zipfile
from collections import Counter
from pathlib import Path
logging.basicConfig(level=logging.INFO, force=True)
import numpy as np, torch
from eviground.config import load_config
from eviground.data import brgen, chameleon, pack
from eviground.data.cp1 import _read_csv_gz
from eviground.env import get_paths, session_info
from eviground.eval import p6data, runner, subsets, testsets
from eviground.explain import cp5, evidence, faithfulness
from eviground.utils import session

SEED = 42
CU_PER_HOUR = 2.2          # the measured L4 rate (18 Nov 2026): Colab's balance fell ~2.2 CU per hour
CU_BALANCE_START = None    # REQUIRED: type the balance Colab shows now (the real cost is the drop to the next balance)
EST_CU = 1.5               # estimate ~1.2 (staging + cache + a few minutes of compute); 1.5 for margin
EXPERIMENT = 'eviground_ctx'
N_VAL_EDITED, N_VAL_RAND, N_P6_RAND = 200, 30, 50
N_OVERLAY = {'edited': 10, 'generated': 5, 'real': 5}
STEPS = faithfulness.STEPS  # deletion / insertion steps
INDEX = 'index.parquet'    # the shard index
T0 = time.time()
info = session_info(cu_per_hour=CU_PER_HOUR)
paths = get_paths()
if DRY:
    from eviground.env import Paths
    _dry = Path(tempfile.gettempdir()) / 'eviground_cp5_dry'
    shutil.rmtree(_dry, ignore_errors=True)
    paths = Paths(root=_dry / 'drive' / 'Paper 1', local_data=_dry / 'data')
    N_VAL_EDITED, N_VAL_RAND, N_P6_RAND, STEPS, EST_CU, CU_BALANCE_START = 3, 2, 2, 4, 0.0, 10.0
    N_OVERLAY = {'edited': 2, 'generated': 1, 'real': 1}
paths.ensure()
CONFIG = 'debug' if DRY else 'base'
HF_CACHE = paths.cache / 'hf' if DRY else Path('/content/hf_cache')
OVERRIDES = [f'model.semantic.cache_dir={HF_CACHE}', f'compute.cu_per_hour={CU_PER_HOUR}']
cfg = load_config(CONFIG, OVERRIDES)
if not DRY:
    assert torch.cuda.is_available(), 'needs a GPU runtime (L4)'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP = lambda: runner.autocast(DEVICE)  # noqa: E731  (bf16 on the L4)
CK = paths.runs / 'checkpoints'
CP1_MANIFEST = CK / 'cp1' / 'manifest_cp1.csv.gz'
SUBSETS_FILE = CK / 'eval_subsets.json'
RUN = paths.runs / f'{EXPERIMENT}_unified_seed{SEED}'
BEST = RUN / 'best.pt'
OUT = paths.runs / 'cp5'
BUNDLES = paths.runs / 'explain' / f'p6_unified_ctx_seed{SEED}'
JSONL = BUNDLES / 'p6_records.jsonl'
BRGEN_ARCHIVE = paths.datasets / '04_BR-Gen' / brgen.LAST_VOLUME
CHAM_ZIP = paths.datasets / '05_Chameleon' / chameleon.ZIP_NAME
RESULT = OUT / f'cp5_token_deletion_seed{SEED}.json'
TOKEN_JSONL = OUT / f'p6_token_deletion_records_seed{SEED}.jsonl'
CACHE_TAR = paths.runs / 'cache' / 'p6_images.tar'
CACHE_MAN = paths.runs / 'cache' / 'p6_images_manifest.json'
N_RANDOM = cp5.N_RANDOM
if not DRY and CU_BALANCE_START is None:
    raise ValueError('type the Colab balance into CU_BALANCE_START (cell 2) before Run all')
OUT.mkdir(parents=True, exist_ok=True); BUNDLES.mkdir(parents=True, exist_ok=True)
print('commit', info['git_commit'], '| gpu', info.get('gpu_name'), '| device', DEVICE, '| model', BEST)

In [ ]:
# Cell 3: shards (validation images) and, in the dry run, the fake world
SHARDS = paths.shards
if DRY:   # fake world -> CP1 -> shards -> a tiny unified model (one epoch); a small fake P6
    from eviground.data.cp1 import CP1Config, run_cp1
    from eviground.data.download import ensure_data
    from eviground.data.fake_cp1 import cp1_overrides, make_cp1_world
    from eviground.train.trainer import Trainer
    sources = make_cp1_world(paths.datasets, paths.root.parent / 'fake_remote', coco_images=40)
    ensure_data('cp1', paths=paths, sources=sources, stage_local=True, overrides=cp1_overrides(40))
    run_cp1(paths, CP1Config(workers=2, crops_per_class=60, blockiness_per_group=20, base_unclear_max=0.2,
                             diffseg_card_splits={'train': 18, 'validation': 12}),
            out_dir=CK / 'cp1', cache_dir=paths.cache / 'cp1', progress=None)
    pack.pack(CP1_MANIFEST, SHARDS, free_bytes=10**12, margin=0, shard_bytes=200_000, sample_per_source=5,
              progress=None)
    Trainer(load_config(CONFIG, OVERRIDES + ['train.epochs=1', 'data.context_crop_share=0.3333']), SHARDS, RUN,
            seed=SEED, profile='unified', resume=None, log=None).fit()
    import hashlib
    rows = _read_csv_gz(CP1_MANIFEST)
    pick = lambda f, n: sorted((r for r in rows if f(r)), key=lambda r: r['uid'])[:n]  # noqa: E731
    parts = {'diffseg_heldout': pick(lambda r: r['role'] == 'test_P2', 2),
             'sidset_tampered': pick(lambda r: r['role'] == 'test_P3b' and r['label'] == 'edited' and r['mask_path'], 2),
             'generated': pick(lambda r: r['role'] == 'test_P1' and r['label'] == 'generated', 1),
             'real': pick(lambda r: r['role'] == 'test_P1' and r['label'] == 'real', 1)}
    from PIL import Image as _PILImage
    for part in ('sidset_tampered', 'diffseg_heldout'):   # half-size masks, as some real P6 masks are (24 Nov)
        mp = Path(parts[part][0]['mask_path'])
        with _PILImage.open(mp) as _m:
            _m.resize((max(1, _m.width // 2), max(1, _m.height // 2)), _PILImage.NEAREST).save(mp)
    rgb = np.full((300, 280, 3), 90, np.uint8); m = np.zeros((300, 280), np.uint8); m[40:160, 60:200] = 255
    BRGEN_ARCHIVE.parent.mkdir(parents=True, exist_ok=True); CHAM_ZIP.parent.mkdir(parents=True, exist_ok=True)
    br_files = {}
    with zipfile.ZipFile(BRGEN_ARCHIVE, 'w') as zf:
        for i in range(2):
            f, mk = f'BR-Gen/Forged/LaMa/Stuff/COCO/{i}_stuff.png', f'BR-Gen/Mask/Stuff/COCO/{i}_stuff.png'
            fb, mb = p6data.png_bytes(rgb + i), p6data.png_bytes(m)
            zf.writestr(f, fb); zf.writestr(mk, mb)
            br_files[f] = {'sha256': hashlib.sha256(fb).hexdigest(), 'mask': mk, 'mask_sha256': hashlib.sha256(mb).hexdigest()}
    cham = 'Chameleon/test/0_real/r0.png'
    with zipfile.ZipFile(CHAM_ZIP, 'w') as zf:
        zf.writestr(cham, p6data.png_bytes(rgb))
    part_of = {r['uid']: k for k, rs in parts.items() for r in rs}
    part_of.update(dict.fromkeys(br_files, 'brgen')); part_of[cham] = 'real'
    chosen = [{'uid': u} for u in part_of]
    edited_uids = [u for u, k in part_of.items() if k in p6data.EDITED_PARTS]
    P6_FAKE = subsets._record('P6', chosen, SEED, 'dry', part_of=part_of)
    subsets.save({'P6': P6_FAKE, 'P6_mllm': subsets._record('P6_mllm', chosen, SEED, 'dry'),
                  'P6_deletion': subsets._record('P6_deletion', [{'uid': u} for u in edited_uids[:3]], SEED, 'dry'),
                  'P6_brgen_files': {'files': br_files}}, SUBSETS_FILE)
CACHE_READY = CACHE_TAR.is_file() and CACHE_MAN.is_file()
NEEDED = {'unified checkpoint (final configuration, seed 42)': BEST, 'shards': SHARDS / INDEX,
          'CP1 manifest': CP1_MANIFEST, 'eval_subsets.json (P6)': SUBSETS_FILE}
if not DRY:   # the notebook 14 bundles (maps) and, unless the cache exists, the raw archives
    NEEDED['notebook 14 P6 records'] = JSONL
    if not CACHE_READY:   # cell 4 reuses the cache only when both the tar and its manifest exist
        NEEDED.update({'BR-Gen archive': BRGEN_ARCHIVE, 'Chameleon.zip': CHAM_ZIP})
PREFLIGHT = session.preflight(
    {'notebook': '15 CP5 token deletion', 'experiment': EXPERIMENT, 'model': BEST, 'result file': RESULT,
     'P6 cache': CACHE_TAR if CACHE_READY else 'built in this session', 'balance at start': CU_BALANCE_START},
    paths=NEEDED, expect={'experiment': 'eviground_ctx'}, est_cu=EST_CU, cu_left=CU_BALANCE_START)
STORED = json.loads(SUBSETS_FILE.read_text())
for k in ('P6', 'P6_mllm', 'P6_deletion'):
    assert subsets.intact(STORED.get(k, {})), f'{SUBSETS_FILE}: {k} is missing or was changed: stop and ask Dhanush'
P6 = STORED['P6']
LOCAL_SHARDS = paths.local_data / 'shards'
print(pack.stage_shards(SHARDS, LOCAL_SHARDS, progress=print))
SHARDS = LOCAL_SHARDS

In [ ]:
# Cell 4: the P6 images: from the Drive cache if it exists (checked against its SHA-256 manifest),
# otherwise staged as in notebook 14 and saved into the cache for later notebooks
t = time.time()
LOCAL_CACHE = paths.local_data / 'p6_cache'
LOCAL_TAR = paths.local_data / 'p6_images.tar'
CACHE_READY = CACHE_TAR.is_file() and CACHE_MAN.is_file()   # again: a rerun of this cell reuses the cache it built
if CACHE_READY:
    shutil.copyfile(CACHE_TAR, LOCAL_TAR)
    CACHE = p6data.read_cache(LOCAL_TAR, CACHE_MAN, LOCAL_CACHE)
    CACHE_SOURCE = 'reused'
else:
    t = time.time()
    MANIFEST = _read_csv_gz(CP1_MANIFEST)
    ROWS = p6data.manifest_rows(P6, MANIFEST)
    if not DRY:
        from eviground.data.download import ensure_data
        ensure_data(datasets=['diffseg30k', 'sidset'], paths=paths, stage_local=True)
    gi = [r for r in ROWS.values() if r.get('source') == 'genimage']
    for r in testsets.stage_files(gi, paths.local_data / 'p6_genimage'):
        ROWS[r['uid']] = r
    CHAM_DIR = paths.local_data / 'p6_chameleon'
    cham_uids = [u for u in P6['uids'] if u not in ROWS and P6['part_of'][u] == 'real']
    with zipfile.ZipFile(CHAM_ZIP) as zf:
        for u in cham_uids:
            zf.extract(u, CHAM_DIR)
    BR_FILES = STORED['P6_brgen_files']['files']
    BR_DIR = paths.local_data / 'p6_brgen'
    br_members = [u for u in P6['uids'] if P6['part_of'][u] == 'brgen']
    exe = brgen.seven_zip()
    if exe is None and not DRY:
        subprocess.run(['apt-get', 'install', '-y', '-q', 'p7zip-full'], check=True, capture_output=True)
        exe = brgen.seven_zip()
    assert exe, '7z is not available'
    brgen.extract(BRGEN_ARCHIVE, br_members + sorted({BR_FILES[u]['mask'] for u in br_members}), BR_DIR, exe)
    BAD = p6data.check_brgen({u: BR_FILES[u] for u in br_members}, BR_DIR)
    if BAD:
        raise RuntimeError(f'BR-Gen files differ from notebook 12b: {BAD[:5]}: stop and ask Dhanush')
    STAGE_MIN = round((time.time() - t) / 60, 2)
    print(f"P6: {len(ROWS)} manifest images, {len(cham_uids)} Chameleon, {len(br_members)} BR-Gen (hashes checked) "
          f"| staged in {STAGE_MIN} min")
    brgen_masks = {u: BR_FILES[u]['mask'] for u in BR_FILES}
    items = p6data.p6_items(P6['uids'], P6['part_of'], ROWS, brgen_dir=BR_DIR, brgen_masks=brgen_masks,
                            chameleon_dir=CHAM_DIR)
    local_man = paths.local_data / 'p6_images_manifest.json'
    p6data.write_cache(items, LOCAL_TAR, local_man)
    CACHE_TAR.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(LOCAL_TAR, CACHE_TAR); shutil.copyfile(local_man, CACHE_MAN)
    CACHE = p6data.read_cache(LOCAL_TAR, CACHE_MAN, LOCAL_CACHE)
    CACHE_SOURCE = 'built'
assert set(CACHE['entries']) == set(P6['uids']), 'the P6 cache does not hold exactly the frozen P6 images'
CACHE_MIN = round((time.time() - t) / 60, 2)
print(f"P6 cache {CACHE_SOURCE}: {CACHE['images']} images, tar {CACHE['tar_sha256'][:12]} | {CACHE_MIN} min")
MASK_RESIZE = p6data.resize_summary(rec['meta'] for rec in CACHE['entries'].values())
for part, r in MASK_RESIZE.items():
    if part in p6data.EDITED_PARTS:
        print(f"  {part:16} masks resized to their image: {r['resized']} of {r['items']} "
              f"(aspect ratio off > 2%: {r['aspect_mismatch']}, left out of mask metrics)  from {r['from_sizes']}")

In [ ]:
# Cell 5: the model
model = runner.load_trained(cfg.model, BEST, DEVICE)
print('model loaded:', BEST, '| edit threshold', model.edit_threshold)

if DRY:   # the fake P6 needs the bundles notebook 14 would have written
    for item in p6data.cached_items(P6['uids'], CACHE, LOCAL_CACHE):
        out = cp5.explain_image(model, item.image, DEVICE, AMP)
        rec, rles = evidence.evidence_record(out['probs'], out['gate'], out['decoder'], out['gradcam_pred'], item.image)
        evidence.save_bundle(BUNDLES, item.uid, rec, rles, {'decoder': out['decoder'], 'gradcam': out['gradcam'],
                             'gradcam_pred': out['gradcam_pred']}, out['probs'], out['gate'])

In [ ]:
# Cell 6: token-level deletion on the 300 P6_deletion images (best tile; maps from the notebook 14 bundles). Resumes.
from eviground.data import tiles as tiling
DELETION = sorted(STORED['P6_deletion']['uids'])
done = set()
if TOKEN_JSONL.is_file():
    done = {json.loads(line)['uid'] for line in TOKEN_JSONL.read_text().splitlines() if line.strip()}
todo = [u for u in DELETION if u not in done]
print(f'token deletion: {len(done)} images done earlier, {len(todo)} to go')
t = time.time()
with open(TOKEN_JSONL, 'a') as fh:
    for item in p6data.cached_items(todo, CACHE, LOCAL_CACHE):
        maps = evidence.load_bundle(BUNDLES, item.uid)['maps']
        h, w = item.mask.shape
        box = faithfulness.best_tile(item.mask, tiling.tile_boxes(h, w))
        le, tp, ri, bo = box
        scores = {'oracle': cp5.token_scores(item.mask[tp:bo, le:ri].astype(float)),
                  'decoder': cp5.token_scores(cp5.bundle_tile(maps['decoder'], box)),
                  'gradcam': cp5.token_scores(cp5.bundle_tile(maps['gradcam'], box))}
        tile = cp5.to_tensor(item.image, DEVICE)[:, tp:bo, le:ri]
        rec = cp5.token_deletion(model, tile, scores, steps=STEPS, n_random=N_RANDOM, seed=SEED, amp=AMP)
        fh.write(json.dumps({'uid': item.uid, 'part': item.meta.get('part'),
                             'tile_edited_share': float(item.mask[tp:bo, le:ri].mean()), **rec}) + '\n'); fh.flush()
TOKEN_RECORDS = [json.loads(line) for line in TOKEN_JSONL.read_text().splitlines() if line.strip()]
assert {r['uid'] for r in TOKEN_RECORDS} == set(DELETION), 'token deletion records are incomplete'
TOKEN_MIN = round((time.time() - t) / 60, 2)
TOKEN = {'all': cp5.token_deletion_verdict(TOKEN_RECORDS)}
for part in p6data.EDITED_PARTS:
    TOKEN[part] = cp5.token_deletion_verdict([r for r in TOKEN_RECORDS if r['part'] == part])
v = TOKEN['all']
print(f"validity: oracle {v['validity']['mean_oracle_auc']:.3f} vs random {v['validity']['mean_random_auc']:.3f} "
      f"(p {v['validity']['wilcoxon_p']:.2g}) -> {'VALID' if v['validity']['valid'] else 'UNINFORMATIVE'}")
for name, m in v['maps'].items():
    print(f"  {name:8} AUC {m['mean_auc']:.3f} vs random {m['mean_random_auc']:.3f} (p {m['wilcoxon_p']:.2g}) -> {m['verdict']}")

In [ ]:
# Cell 7: randomization again on the 30 validation and 50 P6 tiles of notebook 14: SSIM (criterion) and Spearman (reported)
from eviground.data.pack import read_index
val = [r for r in read_index(SHARDS / INDEX) if r['split'] == 'val']
by = [r for r in val if r['label'] == 'edited']
if DRY and not by:
    by = [r for r in read_index(SHARDS / INDEX) if r['split'] == 'train' and r['label'] == 'edited']
V_EDITED = subsets.take(by, N_VAL_EDITED, SEED, 'cp5_val_v1/edited')   # the notebook 14 selection

def best_tiles(items, n):
    out = []
    for item in items:
        if item.mask is None or not item.mask.any():
            continue
        h, w = item.mask.shape
        le, tp, ri, bo = faithfulness.best_tile(item.mask, tiling.tile_boxes(h, w))
        out.append(cp5.to_tensor(item.image, DEVICE)[:, tp:bo, le:ri])
        if len(out) == n:
            break
    return torch.stack(out)

RAND = {'validation': cp5.randomization(model, best_tiles(testsets.shard_items(SHARDS, V_EDITED), N_VAL_RAND), seed=SEED, amp=AMP),
        'p6': cp5.randomization(model, best_tiles(p6data.cached_items(DELETION[:N_P6_RAND], CACHE, LOCAL_CACHE), N_P6_RAND), seed=SEED, amp=AMP)}
RAND_SUMMARY = {where: {name: [{'step': s['step'], 'ssim_mean': s['ssim_mean'], 'spearman_mean': s['spearman_mean']}
                               for s in steps] for name, steps in r.items()} for where, r in RAND.items()}
for where, r in RAND_SUMMARY.items():
    for name, steps in r.items():
        last = steps[-1]
        print(f"{where:10} {name:8} last step: SSIM {last['ssim_mean']:.3f} ({'< 0.5 pass' if last['ssim_mean'] < 0.5 else 'fail'}), "
              f"Spearman {last['spearman_mean']}")

In [ ]:
# Cell 8: result file, per-image CSV and the mean curves
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
RESULT_DATA = {
    'step': 'CP5 repair: token-level deletion before fusion (pre-registered, 24 Nov 2026)', 'dry_run': DRY,
    'branch': BRANCH, 'seed': SEED, 'experiment': EXPERIMENT, 'session': info,
    'rule': {'removal': 'semantic patch + trace tokens -> per-tile mean tokens at the removed grid positions; '
                        'fusion + gate + class head re-run; CLIP class token kept',
             'random_orders': N_RANDOM, 'steps': STEPS, 'p_max': cp5.P_MAX,
             'validity': 'oracle AUC below random, one-sided paired Wilcoxon p < 0.01',
             'map_pass': 'map AUC below random, one-sided paired Wilcoxon p < 0.01',
             'randomization': 'SSIM < 0.5 is the criterion; Spearman reported beside it'},
    'token_deletion': TOKEN, 'randomization': RAND_SUMMARY, 'primary_map': 'decoder',
    'p6_cache': {'source': CACHE_SOURCE, 'tar_sha256': CACHE['tar_sha256'], 'images': CACHE['images'], 'minutes': CACHE_MIN},
    'p6_mask_resize': {'rule': 'mask of another size -> torch interpolate (nearest) to the image, > 0; '
                               'aspect ratio off by more than 2% stops', 'parts': MASK_RESIZE},
    'token_minutes_this_session': TOKEN_MIN,
    'utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
}
RESULT.write_text(json.dumps(RESULT_DATA, indent=1, default=str))
keys = ['uid', 'part', 'tile_edited_share', 'token_p_full', 'token_oracle', 'token_decoder', 'token_gradcam',
        'token_random', 'token_random_sd']
with open(OUT / f'p6_token_deletion_seed{SEED}.csv', 'w', newline='') as fh:
    w = csv.DictWriter(fh, fieldnames=keys, extrasaction='ignore'); w.writeheader(); w.writerows(TOKEN_RECORDS)
fig, ax = plt.subplots(figsize=(5, 4))
xs = np.linspace(0, 1, STEPS + 1)
for name in cp5.TOKEN_ORDERS:
    ax.plot(xs, np.mean([r[f'token_{name}_curve'] for r in TOKEN_RECORDS], axis=0), label=name)
ax.set_xlabel('share of tokens removed'); ax.set_ylabel('mean P(edited)'); ax.legend(); fig.tight_layout()
fig.savefig(OUT / 'cp5_token_curves.png', dpi=120); plt.close(fig)
print('saved', RESULT)
if DRY:
    assert set(TOKEN['all']['maps']) == {'decoder', 'gradcam'}
    assert RAND_SUMMARY['p6']['decoder'][-1]['step'] == 'semantic_lora'
    assert MASK_RESIZE['sidset_tampered']['resized'] >= 1 and MASK_RESIZE['diffseg_heldout']['resized'] >= 1

In [ ]:
# Cell 9: compute record (with the starting balance), then release the runtime
COMPUTE = session.finish(OUT, T0, CU_PER_HOUR, gpu=info.get('gpu_name'), name='compute_cp5_token.json', balance_start=CU_BALANCE_START,
                         extra={'git_commit': info['git_commit'], 'est_cu': EST_CU})
print('Send Claude the balance Colab shows after this cell: the run cost is the drop from', CU_BALANCE_START,
      '(recorded later with session.record_balance)')
RELEASED = session.release(DRY)
if DRY:
    assert RELEASED == 'dry', RELEASED